In [1]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold

DATA_DIR = Path("../Data/Raw")

# Reload the fully encoded datasets from Step 1
df_german = pd.read_csv(DATA_DIR / "German Credit Risk Dataset" / "german_credit_encoded.csv")
df_home = pd.read_csv(DATA_DIR / "Home Credit Default Risk Dataset" / "application_train_encoded.csv")
df_lending = pd.read_csv(DATA_DIR / "Lending Club Loan Dataset" / "loans_full_schema_encoded.csv")

# Standardize target column name across all three datasets
df_home = df_home.rename(columns={"TARGET": "target"})

for name, df in [("German Credit", df_german), ("Home Credit", df_home), ("LendingClub", df_lending)]:
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns, target present: {'target' in df.columns}")

German Credit: 1000 rows, 41 columns, target present: True
Home Credit: 307507 rows, 102 columns, target present: True
LendingClub: 10000 rows, 70 columns, target present: True


In [2]:
# Split each dataset into X (features) and y (target)
# SK_ID_CURR (Home Credit's ID column) is dropped - it's an identifier, not a feature
X_german = df_german.drop(columns=["target"])
y_german = df_german["target"]

X_home = df_home.drop(columns=["target", "SK_ID_CURR"])
y_home = df_home["target"]

X_lending = df_lending.drop(columns=["target"])
y_lending = df_lending["target"]

print("German:", X_german.shape, y_german.shape)
print("Home:", X_home.shape, y_home.shape)
print("Lending:", X_lending.shape, y_lending.shape)

German: (1000, 40) (1000,)
Home: (307507, 100) (307507,)
Lending: (10000, 69) (10000,)


In [3]:
# Fixed seed used throughout the project, for reproducibility (our "seed register")
RANDOM_SEED = 42

X_german_train, X_german_test, y_german_train, y_german_test = train_test_split(
    X_german, y_german, test_size=0.2, stratify=y_german, random_state=RANDOM_SEED
)

X_home_train, X_home_test, y_home_train, y_home_test = train_test_split(
    X_home, y_home, test_size=0.2, stratify=y_home, random_state=RANDOM_SEED
)

X_lending_train, X_lending_test, y_lending_train, y_lending_test = train_test_split(
    X_lending, y_lending, test_size=0.2, stratify=y_lending, random_state=RANDOM_SEED
)

# Confirm the split preserved class balance (stratify should guarantee this)
for name, y_tr, y_te in [
    ("German", y_german_train, y_german_test),
    ("Home", y_home_train, y_home_test),
    ("Lending", y_lending_train, y_lending_test)
]:
    print(f"\n{name} - Train shape: {y_tr.shape}, Test shape: {y_te.shape}")
    print(f"{name} - Train class balance:\n{y_tr.value_counts(normalize=True)}")
    print(f"{name} - Test class balance:\n{y_te.value_counts(normalize=True)}")


German - Train shape: (800,), Test shape: (200,)
German - Train class balance:
target
0    0.7
1    0.3
Name: proportion, dtype: float64
German - Test class balance:
target
0    0.7
1    0.3
Name: proportion, dtype: float64

Home - Train shape: (246005,), Test shape: (61502,)
Home - Train class balance:
target
0    0.91927
1    0.08073
Name: proportion, dtype: float64
Home - Test class balance:
target
0    0.919271
1    0.080729
Name: proportion, dtype: float64

Lending - Train shape: (8000,), Test shape: (2000,)
Lending - Train class balance:
target
0    0.98225
1    0.01775
Name: proportion, dtype: float64
Lending - Test class balance:
target
0    0.982
1    0.018
Name: proportion, dtype: float64


In [4]:
# 5-fold stratified CV, same seed as the train/test split, applied to training data only
# One StratifiedKFold object can be reused across all three datasets - it's not
# tied to a specific dataset until you call .split(X, y) on it
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

# Save fold assignments for German Credit as an example - this becomes part of
# the audit trail, proving exactly which rows were in which fold for reproducibility
german_folds = list(skf.split(X_german_train, y_german_train))
print(f"Number of folds created: {len(german_folds)}")
print(f"Fold 1 - train size: {len(german_folds[0][0])}, validation size: {len(german_folds[0][1])}")

Number of folds created: 5
Fold 1 - train size: 640, validation size: 160


In [5]:
import numpy as np

FOLDS_DIR = Path("../logs/fold_assignments")
FOLDS_DIR.mkdir(parents=True, exist_ok=True)

# Generate and save fold indices for each dataset separately
# (each dataset has its own training set, so its own fold splits)
datasets_for_cv = {
    "german": (X_german_train, y_german_train),
    "home": (X_home_train, y_home_train),
    "lending": (X_lending_train, y_lending_train)
}

for name, (X_tr, y_tr) in datasets_for_cv.items():
    folds = list(skf.split(X_tr, y_tr))
    fold_data = {f"fold_{i}": {"train_idx": train_idx.tolist(), "val_idx": val_idx.tolist()}
                 for i, (train_idx, val_idx) in enumerate(folds)}
    
    import json
    with open(FOLDS_DIR / f"{name}_folds.json", "w") as f:
        json.dump(fold_data, f)
    print(f"Saved fold assignments for {name}: {len(folds)} folds")

Saved fold assignments for german: 5 folds
Saved fold assignments for home: 5 folds
Saved fold assignments for lending: 5 folds


In [6]:
from sklearn.preprocessing import MinMaxScaler

# Fit the scaler on TRAINING data only, then apply (transform) to both train and test.
# This is the correct order to avoid data leakage: if we fit on the full dataset
# (including test data) before splitting, information about the test set's
# range would leak into how training data gets scaled.

def scale_train_test(X_train, X_test):
    scaler = MinMaxScaler()
    X_train_scaled = pd.DataFrame(
        scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index
    )
    X_test_scaled = pd.DataFrame(
        scaler.transform(X_test), columns=X_test.columns, index=X_test.index
    )
    return X_train_scaled, X_test_scaled, scaler

X_german_train_scaled, X_german_test_scaled, scaler_german = scale_train_test(X_german_train, X_german_test)
X_home_train_scaled, X_home_test_scaled, scaler_home = scale_train_test(X_home_train, X_home_test)
X_lending_train_scaled, X_lending_test_scaled, scaler_lending = scale_train_test(X_lending_train, X_lending_test)

# Confirm scaling worked - all values should now be between 0 and 1
print("German train range:", X_german_train_scaled.min().min(), "to", X_german_train_scaled.max().max())
print("Home train range:", X_home_train_scaled.min().min(), "to", X_home_train_scaled.max().max())
print("Lending train range:", X_lending_train_scaled.min().min(), "to", X_lending_train_scaled.max().max())

# Important: test set values can fall slightly outside [0,1] - that's expected
# and correct, since the scaler was fit on training data's min/max, not the test set's
print("\nGerman test range:", X_german_test_scaled.min().min(), "to", X_german_test_scaled.max().max())
print("Home test range:", X_home_test_scaled.min().min(), "to", X_home_test_scaled.max().max())
print("Lending test range:", X_lending_test_scaled.min().min(), "to", X_lending_test_scaled.max().max())

German train range: 0.0 to 1.0
Home train range: 0.0 to 1.0000000000000002
Lending train range: 0.0 to 1.0000000000000002

German test range: -0.0034181854036909806 to 1.0
Home test range: 0.0 to 1.0000000000000002
Lending test range: 0.0 to 1.5942028985507246


In [7]:
# Identify which column(s) in the LendingClub test set exceed the [0,1] range
# by a meaningful amount, not just floating-point rounding
test_max_per_col = X_lending_test_scaled.max()
print(test_max_per_col[test_max_per_col > 1.01].sort_values(ascending=False))

annual_income_joint             1.594203
current_installment_accounts    1.060606
dtype: float64


In [8]:
SCALED_DIR = Path("../Data/Processed")
SCALED_DIR.mkdir(parents=True, exist_ok=True)

X_german_train_scaled.to_csv(SCALED_DIR / "german_X_train.csv", index=False)
X_german_test_scaled.to_csv(SCALED_DIR / "german_X_test.csv", index=False)
y_german_train.to_csv(SCALED_DIR / "german_y_train.csv", index=False)
y_german_test.to_csv(SCALED_DIR / "german_y_test.csv", index=False)

X_home_train_scaled.to_csv(SCALED_DIR / "home_X_train.csv", index=False)
X_home_test_scaled.to_csv(SCALED_DIR / "home_X_test.csv", index=False)
y_home_train.to_csv(SCALED_DIR / "home_y_train.csv", index=False)
y_home_test.to_csv(SCALED_DIR / "home_y_test.csv", index=False)

X_lending_train_scaled.to_csv(SCALED_DIR / "lending_X_train.csv", index=False)
X_lending_test_scaled.to_csv(SCALED_DIR / "lending_X_test.csv", index=False)
y_lending_train.to_csv(SCALED_DIR / "lending_y_train.csv", index=False)
y_lending_test.to_csv(SCALED_DIR / "lending_y_test.csv", index=False)

print("All scaled train/test files saved to Data/Processed/")

All scaled train/test files saved to Data/Processed/
